# Evaluate Fine-tune EmbeddingGemma ZeroShot Classification

## not zero shot -> supervised metric learning

Fine-tuning helps close the gap between a model's general-purpose understanding and the specialized, high-performance accuracy that your application requires. Since no single model is perfect for every task, fine-tuning adapts it to your specific domain.

### Install Python packages

Install the libraries required for running the EmbeddingGemma model and generating embeddings. Sentence Transformers is a Python framework for text and image embeddings. For more information, see the [Sentence Transformers](https://www.sbert.net/) documentation.

In [ ]:
!pip install -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-ycah5n6v
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-ycah5n6v
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.7 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-4.57.0.dev0-py3-none-any.whl size=12604658 sha256=b3a135b780e2e9a3a389c7e493e8fbd18b5a0eee236c94d52d4bbaf4af361ff5
  Stored in directory: /tmp/pip-ephem-wheel-cache-4rzcoo2k/wheels/3a/21/76/c31899bac2cf601d3c74091b26a413bc3fb54770d5ccb5c924
Successfully built transformers
  Attempting uni

After you have accepted the license, you need a valid Hugging Face Token to access the model.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('DUSH_HFT')
# WANDB_TOKEN = userdata.get('DUSH_WANDB')

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login
login(token=HF_TOKEN)

# import wandb
# wandb.login(key=WANDB_TOKEN)

## Prepare the Fine-Tuning Dataset

This is the most crucial part. You need to create a dataset that teaches the model what "similar" means in your specific context. This data is often structured as triplets: (anchor, positive, negative)

- Anchor: The original query or sentence.
- Positive: A sentence that is semantically very similar or identical to the anchor.
- Negative: A sentence that is on a related topic but semantically distinct.

In this example, we only prepared 3 triplets, but for a real application, you would need a much larger dataset to perform well.

In [ ]:
from datasets import Dataset
from datasets import load_dataset
import torch

ds_test = load_dataset("dinushiTJ/nz_hansard_inference_results", split="test")
ds_test

TASK_NAME = "Classification"

CONTEXT_LEN = 2048  # Gemma context length
EMBED_DIM = 768     # Gemma embedding size
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.90M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/213 [00:00<?, ? examples/s]

In [ ]:
ds_test

Dataset({
    features: ['HpsProceedingHeading', 'HpsProceedingContent', 'classification_label', 'classification_confidence', 'classification_reason', '_filename', '_path', 'content_char_len', 'content_word_len', 'label', 'token_count', 'conversations', 'BaseResponse', 'FTResponse', 'Base4BResponse', 'FT4BResponse'],
    num_rows: 213
})

In [ ]:
DEVICE

'cuda'

In [ ]:
ds_test[0]['classification_label']

'maori_origin'

In [ ]:
label_names = list(sorted(set(ds_test["classification_label"])))
label_names

['maori_origin', 'non_maori_origin']

In [ ]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

base_model = SentenceTransformer("google/embeddinggemma-300M").to(DEVICE)
base_model.eval()

ft_model = SentenceTransformer("dinushiTJ/nz-hansard-embedding-gemma-zeroshot").to(DEVICE)
ft_model.eval()

base_label_embeddings = base_model.encode(
    label_names,
    prompt_name=TASK_NAME,
    convert_to_numpy=True,
    normalize_embeddings=True
)

ft_label_embeddings = ft_model.encode(
    label_names,
    prompt_name=TASK_NAME,
    convert_to_numpy=True,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

In [ ]:
base_label_embeddings

array([[-0.19262542, -0.03625869,  0.02727376, ..., -0.03909068,
        -0.01297867, -0.00118841],
       [-0.20961605, -0.04762118,  0.01081612, ..., -0.02479456,
        -0.02834242, -0.006023  ]], dtype=float32)

In [ ]:
base_label_embeddings.shape

(2, 768)

In [ ]:
ft_label_embeddings

array([[-0.0099614 ,  0.04461319,  0.10121232, ..., -0.03481654,
         0.10190711, -0.06286851],
       [-0.08007345, -0.02560342, -0.07939161, ...,  0.00615883,
        -0.08359997, -0.01664898]], dtype=float32)

In [ ]:
ft_label_embeddings.shape

(2, 768)

In [ ]:
def embed_document(text: str, token_count: int, model) -> np.ndarray:
    if token_count <= CONTEXT_LEN:
        emb = model.encode(
            text,
            prompt_name=TASK_NAME,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        return emb

    # Chunking path
    words = text.split()
    chunk_size = max(1, int(len(words) * CONTEXT_LEN / token_count))

    chunk_embeddings = []

    for i in range(0, len(words), chunk_size):
        chunk_text = " ".join(words[i:i + chunk_size])
        emb = model.encode(
            chunk_text,
            prompt_name=TASK_NAME,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        chunk_embeddings.append(emb)

    # Mean pool chunk embeddings
    return np.mean(np.stack(chunk_embeddings, axis=0), axis=0)


In [ ]:
def zero_shot_classify_base(example):
    doc_embedding = embed_document(
        text=example["HpsProceedingContent"],
        token_count=example["token_count"],
        model=base_model,
    )

    # Cosine similarity via dot product because embeddings are normalized
    similarities = np.dot(doc_embedding, base_label_embeddings.T)

    best_idx = int(np.argmax(similarities))
    predicted_label = label_names[best_idx]

    return {
        "BaseEmbGemZeroLabel": predicted_label
    }


In [ ]:
ds_test = ds_test.map(
    zero_shot_classify_base,
    desc="Zero-shot classification with Base EmbeddingGemma"
)


Zero-shot classification with Base EmbeddingGemma:   0%|          | 0/213 [00:00<?, ? examples/s]

In [ ]:
for i in range(5):
    print(
        ds_test[i]["classification_label"],
        "->",
        ds_test[i]["BaseEmbGemZeroLabel"]
    )


maori_origin -> maori_origin
non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin
maori_origin -> maori_origin


In [ ]:
def zero_shot_classify_ft(example):
    doc_embedding = embed_document(
        text=example["HpsProceedingContent"],
        token_count=example["token_count"],
        model=ft_model,
    )

    # Cosine similarity via dot product because embeddings are normalized
    similarities = np.dot(doc_embedding, ft_label_embeddings.T)

    best_idx = int(np.argmax(similarities))
    predicted_label = label_names[best_idx]

    return {
        "FTEmbGemZeroLabel": predicted_label
    }


In [ ]:
ds_test = ds_test.map(
    zero_shot_classify_ft,
    desc="Zero-shot classification with FT EmbeddingGemma"
)


Zero-shot classification with FT EmbeddingGemma:   0%|          | 0/213 [00:00<?, ? examples/s]

In [ ]:
for i in range(5):
    print(
        ds_test[i]["classification_label"],
        "->",
        ds_test[i]["FTEmbGemZeroLabel"]
    )


maori_origin -> maori_origin
non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin
maori_origin -> maori_origin


In [ ]:
ds_test.features

{'HpsProceedingHeading': Value('string'),
 'HpsProceedingContent': Value('string'),
 'classification_label': Value('string'),
 'classification_confidence': Value('float64'),
 'classification_reason': Value('string'),
 '_filename': Value('string'),
 '_path': Value('string'),
 'content_char_len': Value('int64'),
 'content_word_len': Value('int64'),
 'label': ClassLabel(names=['maori_origin', 'non_maori_origin']),
 'token_count': Value('int64'),
 'conversations': List({'content': Value('string'), 'role': Value('string')}),
 'BaseResponse': Value('string'),
 'FTResponse': Value('string'),
 'Base4BResponse': Value('string'),
 'FT4BResponse': Value('string'),
 'BaseEmbGemZeroLabel': Value('string'),
 'FTEmbGemZeroLabel': Value('string')}

In [ ]:
ds_test.push_to_hub("dinushiTJ/nz_hansard_inference_results")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  88%|########7 | 2.57MB / 2.93MB            

CommitInfo(commit_url='https://huggingface.co/datasets/dinushiTJ/nz_hansard_inference_results/commit/ce31023ed932cbf2dcfc4996d9369a509011ec9c', commit_message='Upload dataset', commit_description='', oid='ce31023ed932cbf2dcfc4996d9369a509011ec9c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dinushiTJ/nz_hansard_inference_results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dinushiTJ/nz_hansard_inference_results'), pr_revision=None, pr_num=None)